# Session 13 · Data Processing Techniques in ML

Run each cell with **Shift+Enter**. Original material by Aseef Ahmed.

## Example 1 - a messy dataset and a first look

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import MinMaxScaler, StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

df = pd.DataFrame({
    "age": [22, 25, None, 31, 28, 45, 38, 29, 26, 120],
    "city": ["Karachi", "Lahore", "Karachi", None, "Quetta", "Lahore", "Karachi", "Karachi", "Lahore", "Quetta"],
    "education": ["BS", "MS", "BS", "PhD", "Inter", "MS", "BS", "MS", "Inter", "BS"],
    "salary_k": [60, 95, 70, 150, 40, 130, 110, None, 45, 80],
    "hired": [0, 1, 0, 1, 0, 1, 1, 1, 0, 0],
})
print(df.info())
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 5 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   age        9 non-null      float64
 1   city       9 non-null      str    
 2   education  10 non-null     str    
 3   salary_k   9 non-null      float64
 4   hired      10 non-null     int64  
dtypes: float64(2), int64(1), str(2)
memory usage: 619.0 bytes
None
age          1
city         1
education    0
salary_k     1
hired        0
dtype: int64


## Example 2 - missing values: drop vs fill

In [2]:
print("rows if we drop:", len(df.dropna()), "of", len(df))
filled = df.copy()
filled["age"] = filled["age"].fillna(filled["age"].median())
filled["salary_k"] = filled["salary_k"].fillna(filled["salary_k"].mean())
filled["city"] = filled["city"].fillna(filled["city"].mode()[0])
print(filled.isna().sum().sum(), "missing values left")

rows if we drop: 7 of 10
0 missing values left


## Example 3 - outliers with the IQR rule

In [3]:
q1, q3 = filled["age"].quantile([0.25, 0.75]); iqr = q3 - q1
lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print(f"fences: {lo:.1f} .. {hi:.1f}")
print(filled[(filled["age"] < lo) | (filled["age"] > hi)])
filled["age"] = filled["age"].clip(lo, hi)          # cap instead of deleting

fences: 11.9 .. 50.9
     age    city education  salary_k  hired
9  120.0  Quetta        BS      80.0      0


## Example 4 - encoding categories

In [4]:
print(pd.get_dummies(filled, columns=["city"], dtype=int).head(3))
edu_order = [["Inter", "BS", "MS", "PhD"]]
filled["edu_level"] = OrdinalEncoder(categories=edu_order).fit_transform(filled[["education"]])
print(filled[["education", "edu_level"]].drop_duplicates())

    age education  salary_k  hired  city_Karachi  city_Lahore  city_Quetta
0  22.0        BS      60.0      0             1            0            0
1  25.0        MS      95.0      1             0            1            0
2  29.0        BS      70.0      0             1            0            0
  education  edu_level
0        BS        1.0
1        MS        2.0
3       PhD        3.0
4     Inter        0.0


## Example 5 - scaling numbers

In [5]:
nums = filled[["age", "salary_k"]]
print(pd.DataFrame(MinMaxScaler().fit_transform(nums), columns=nums.columns).round(2).head())
print(pd.DataFrame(StandardScaler().fit_transform(nums), columns=nums.columns).round(2).describe().loc[["mean", "std"]])

    age  salary_k
0  0.00      0.18
1  0.10      0.50
2  0.24      0.27
3  0.31      1.00
4  0.21      0.00
           age      salary_k
mean  0.000000  1.665335e-17
std   1.054272  1.053850e+00


## Example 6 - split FIRST, then fit the processing on training data only

In [6]:
X, y = df.drop(columns="hired"), df["hired"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
print(X_train.shape, X_test.shape)

(7, 4) (3, 4)


## Example 7 - one reusable Pipeline + ColumnTransformer

In [7]:
numeric = ["age", "salary_k"]
nominal = ["city"]
ordinal = ["education"]
prep = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numeric),
    ("nom", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(handle_unknown="ignore"))]), nominal),
    ("ord", OrdinalEncoder(categories=edu_order), ordinal),
])
Xtr = prep.fit_transform(X_train)        # learn medians, means, categories from TRAIN
Xte = prep.transform(X_test)             # only apply them to TEST
print(prep.get_feature_names_out())
print(np.round(Xtr, 2))

['num__age' 'num__salary_k' 'nom__city_Karachi' 'nom__city_Lahore'
 'nom__city_Quetta' 'ord__education']
[[-0.48  1.55  1.    0.    0.    3.  ]
 [ 2.4  -0.48  0.    0.    1.    1.  ]
 [-0.37 -0.77  1.    0.    0.    1.  ]
 [-0.26  0.39  1.    0.    0.    1.  ]
 [-0.03  0.97  0.    1.    0.    2.  ]
 [-0.68 -0.04  0.    1.    0.    2.  ]
 [-0.58 -1.64  0.    0.    1.    0.  ]]
